# Customer Churn Prediction using Logistic Regression

This notebook completes the assignment on predicting telecom customer churn using the Telco Customer Churn dataset.

- Objective: build a Logistic Regression model to predict churn.
- Dataset: Telco Customer Churn from Kaggle.
- Output: evaluation metrics, confusion matrix, observations, and conclusion.

In [ ]:
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## Task 2: Data Preprocessing

- Check for missing values.
- Handle missing values using imputation.
- Encode categorical variables using one-hot encoding.
- Split the dataset into 80% training and 20% testing.

In [ ]:
def load_dataset() -> pd.DataFrame:
    candidates = [
        Path("Telco-Customer-Churn.csv"),
        Path("WA_Fn-UseC_-Telco-Customer-Churn.csv"),
    ]

    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate)

    dataset_dir = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
    csv_candidates = sorted(dataset_dir.rglob("*.csv"))

    if not csv_candidates:
        raise FileNotFoundError(
            "Kaggle dataset was downloaded, but no CSV file was found inside the extracted folder."
        )

    return pd.read_csv(csv_candidates[0])


data = load_dataset()
print("First five records:\n")
display(data.head())

numerical_features_raw = data.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features_raw = data.select_dtypes(include=["object"]).columns.tolist()
target_variable = "Churn"

print("Numerical features:", numerical_features_raw)
print("Categorical features:", categorical_features_raw)
print("Target variable:", target_variable)

# Handle the common whitespace issue in TotalCharges and map the target.
data = data.copy()
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
missing_values = data.isna().sum()
missing_values = missing_values[missing_values > 0]
print("\nMissing values:\n", missing_values if not missing_values.empty else "No missing values found.")

data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})

x = data.drop(columns=["customerID", "Churn"])
y = data["Churn"]

numerical_features = x.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = x.select_dtypes(include=["object"]).columns.tolist()

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## Task 3: Model Development

- Build a Logistic Regression model using the selected features.
- Use a preprocessing pipeline to scale numerical features and encode categorical features.
- Train the model on the training set.
- Predict customer churn on the test dataset.

## Task 4: Model Evaluation

- Evaluate the model using Accuracy Score, Precision, Recall, and F1-Score.
- Generate a Confusion Matrix.
- Write observations based on the model performance.

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

model.fit(x_train, y_train)
y_pred = model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print("Confusion Matrix:\n", cm)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Predicted No Churn", "Predicted Churn"],
    yticklabels=["Actual No Churn", "Actual Churn"],
)
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.show()

observations = [
    "Accuracy can look strong because churn datasets are often imbalanced toward non-churn customers.",
    "Recall matters a lot here because missing churn customers can reduce the usefulness of the model for retention planning.",
    "The confusion matrix shows whether the model is better at detecting non-churners than churners.",
]

print("\nObservations:")
for index, observation in enumerate(observations, start=1):
    print(f"{index}. {observation}")

## Task 5: Conclusion

- Key finding: Logistic Regression provides a solid baseline for churn prediction.
- Important churn drivers: tenure, contract type, monthly charges, and service usage patterns.
- Limitation: it assumes a mostly linear relationship, so it may miss complex nonlinear interactions.